# AksaraScan v3.0 — Pelatihan GPU Penuh (YOLOX + CRNN-CTC, Apache-2.0 / MIT)

Notebook ini menjalankan **Tahap 1 -> Tahap 2 -> Tahap 3** pada GPU Google Colab (T4 / L4 / A100) untuk:
1. **Track A (`yolox-tiny P2 imgsz=960`)**: detektor 55 kelas bebas AGPL-3.0 + konversi FP16/INT8 `< 20 MB`.
2. **Track B (`CRNN + BiLSTM + CTC`)**: pengenal baris bebas segmentasi -> `aksara_crnn_ctc.onnx`.
3. **Unduh ONNX Final**: mengunduh model `< 20 MB` hasil latih GPU agar bisa diverifikasi pada Gerbang Rilis lokal sebelum di-push ke GitHub.

In [ ]:
# 1. Verifikasi GPU & ekstrak bundel AksaraScan v3.0 (`aksarascan_colab_bundle.zip`)
!nvidia-smi
import os, zipfile
from google.colab import files

if not os.path.exists("/content/Aksara_Scan/training/train_yolox.py"):
    print("Unggah aksarascan_colab_bundle.zip dari folder Aksara_Scan Anda...")
    up = files.upload()
    zname = next(iter(up.keys()))
    with zipfile.ZipFile(zname, "r") as zf:
        zf.extractall("/content/Aksara_Scan")
%cd /content/Aksara_Scan
!ls -la training app/web_camscanner

In [ ]:
# 2. Pasang dependensi Apache-2.0/MIT (bebas ultralytics AGPL-3.0)
!pip -q install -r training/requirements.txt onnxruntime-gpu
import torch
print("PyTorch:", torch.__version__, "| CUDA:", torch.cuda.is_available(), "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE")

In [ ]:
# 3. Unduh dataset terbuka (OFL fonts, setilanaji, NusaX, Sastra Jawa, jvwiki, Jawa.zip mirror) & buat Synth v2 (15.000 gambar)
!python tools/datasets/download.py --all-ok
!python tools/datasets/convert_local_handwriting.py || true
!python tools/datasets/convert_setilanaji.py
!python tools/datasets/dedupe_and_validate.py
!python tools/synth_v2/corpus_builder.py
!python tools/synth_v2/generate_v2.py --n-train 13500 --n-val 1500 --out-dir dataset/external/synth_v2

In [ ]:
# 4. Track A Stage 1 — Pralatih YOLOX-Tiny P2 imgsz=960 (120 epoch, fliplr=0, flipud=0)
import os
tr_srcs = ["dataset/external/synth_v2"]
for c in ["dataset/external/converted_yolo/jawa_zip", "dataset/external/converted_yolo/hanacaraka"]:
    if os.path.exists(c):
        tr_srcs.append(c)
cmd = f"python training/train_yolox.py --variant yolox-tiny --use-p2 --imgsz 960 --stage 1 --epochs 120 --close-mosaic 15 --mosaic 0.25 --batch 16 --workers 4 --lr0 0.001 --seed 42 --train-sources {' '.join(tr_srcs)} --val-sources dataset/external/synth_v2 --out-dir runs_yolox/stage1_tiny_960_s42"
print(cmd)
!{cmd}

In [ ]:
# 5. Tambang Hard Negatives & Fine-Tuning Stage 2 (lr=0.0002)
import os
os.makedirs("dataset/external/hard_mined", exist_ok=True)
onnx_s1 = "runs_yolox/stage1_tiny_960_s42/yolox-tiny_img960_s42_fp16.onnx"
if not os.path.exists(onnx_s1):
    onnx_s1 = "runs_yolox/stage1_tiny_960_s42/yolox-tiny_img960_s42.onnx"
!python tools/datasets/hard_negative_mine.py --mode self_check --onnx {onnx_s1} --out-dir dataset/external/hard_mined

s2_srcs = ["dataset/external/synth_v2", "dataset/external/hard_mined"]
for c in ["dataset/external/converted_yolo/jawa_zip", "dataset/external/converted_yolo/hanacaraka", "eval/real_train.jsonl"]:
    if os.path.exists(c):
        s2_srcs.append(c)
val_src = "eval/real_val.jsonl" if os.path.exists("eval/real_val.jsonl") else "dataset/external/synth_v2"
cmd2 = f"python training/train_yolox.py --variant yolox-tiny --use-p2 --imgsz 960 --stage 2 --epochs 45 --close-mosaic 10 --mosaic 0.10 --batch 8 --workers 2 --lr0 0.0002 --resume runs_yolox/stage1_tiny_960_s42/yolox-tiny_img960_s42_best.pt --train-sources {' '.join(s2_srcs)} --val-sources {val_src} --seed 42 --out-dir runs_yolox/stage2_finetune"
print(cmd2)
!{cmd2}

In [ ]:
# 6. Track B — Pelatihan CRNN + BiLSTM + CTC Loss (Segmentation-Free)
!python training/train_crnn_track_b.py --epochs 40 --batch 32 --lr 0.001 --out-dir runs_crnn

In [ ]:
# 7. Unduh model ONNX (< 20 MB) hasil latih GPU ke komputer lokal
import glob, os
from google.colab import files
cands = glob.glob("runs_yolox/stage2_finetune/*.onnx") + glob.glob("runs_crnn/*.onnx")
for p in cands:
    sz = os.path.getsize(p) / (1024 * 1024)
    print(f"{p}: {sz:.2f} MB")
    if sz <= 20.0:
        files.download(p)